In [1]:
# import tensorflow.compat.v1 as tf
# tf.disable_v2_behavior() # 禁用 2.0 默认行为，回到 1.x 模式
# from tensorflow.examples.tutorials.mnist import input_data
# mnist = input_data.read_data_sets('MNIST_data', one_hot=True)

# learning_rate = 1e-4
# keep_prob_rate = 0.7 # 
# max_epoch = 2000
# def compute_accuracy(v_xs, v_ys):
#     global prediction
#     y_pre = sess.run(prediction, feed_dict={xs: v_xs, keep_prob: 1})
#     correct_prediction = tf.equal(tf.argmax(y_pre,1), tf.argmax(v_ys,1))
#     accuracy = tf.reduce_mean(tf.cast(correct_prediction, tf.float32))
#     result = sess.run(accuracy, feed_dict={xs: v_xs, ys: v_ys, keep_prob: 1})
#     return result

# def weight_variable(shape):
#     initial = tf.truncated_normal(shape, stddev=0.1)
#     return tf.Variable(initial)

# def bias_variable(shape):
#     initial = tf.constant(0.1, shape=shape)
#     return tf.Variable(initial)

# def conv2d(x, W):
#     # 每一维度  滑动步长全部是 1， padding 方式 选择 same
#     # 提示 使用函数  tf.nn.conv2d
#     return tf.nn.conv2d(x, W, strides=[1, 1, 1, 1], padding='SAME')

# def max_pool_2x2(x):
#     # 滑动步长 是 2步; 池化窗口的尺度 高和宽度都是2; padding 方式 请选择 same
#     # 提示 使用函数  tf.nn.max_pool
#     return tf.nn.max_pool(x, ksize=[1, 2, 2, 1], strides=[1, 2, 2, 1], padding='SAME')
    

# # define placeholder for inputs to network
# xs = tf.placeholder(tf.float32, [None, 784])/255.
# ys = tf.placeholder(tf.float32, [None, 10])
# keep_prob = tf.placeholder(tf.float32)
# x_image = tf.reshape(xs, [-1, 28, 28, 1])

# #  卷积层 1
# ## conv1 layer ##

# W_conv1 = weight_variable([7, 7, 1, 32]) # patch 7x7, in size 1, out size 32
# b_conv1 = bias_variable([32])    
# h_conv1 = tf.nn.relu(conv2d(x_image, W_conv1) + b_conv1)   # 卷积  自己选择 选择激活函数
# h_pool1 =  max_pool_2x2(h_conv1)          # 池化               

# # 卷积层 2
# W_conv2 =   weight_variable([5, 5, 32, 64])       # patch 5x5, in size 32, out size 64
# b_conv2 = bias_variable([64])
# h_conv2 =tf.nn.relu(conv2d(h_pool1, W_conv2) + b_conv2)         # 卷积  自己选择 选择激活函数
# h_pool2 =    max_pool_2x2(h_conv2)        # 池化

# #  全连接层 1
# ## fc1 layer ##
# W_fc1 = weight_variable([7*7*64, 1024])
# b_fc1 = bias_variable([1024])

# h_pool2_flat = tf.reshape(h_pool2, [-1, 7*7*64])
# h_fc1 = tf.nn.relu(tf.matmul(h_pool2_flat, W_fc1) + b_fc1)
# h_fc1_drop = tf.nn.dropout(h_fc1, keep_prob)

# # 全连接层 2
# ## fc2 layer ##
# W_fc2 = weight_variable([1024, 10])
# b_fc2 = bias_variable([10])
# prediction = tf.nn.softmax(tf.matmul(h_fc1_drop, W_fc2) + b_fc2)


# # 交叉熵函数
# cross_entropy = tf.reduce_mean(-tf.reduce_sum(ys * tf.log(prediction),
#                                               reduction_indices=[1]))
# train_step = tf.train.AdamOptimizer(learning_rate).minimize(cross_entropy)

# with tf.Session() as sess:
#     init = tf.global_variables_initializer()
#     sess.run(init)
    
#     for i in range(max_epoch):
#         batch_xs, batch_ys = mnist.train.next_batch(100)
#         sess.run(train_step, feed_dict={xs: batch_xs, ys: batch_ys, keep_prob:keep_prob_rate})
#         if i % 100 == 0:
#             print(compute_accuracy(
#                 mnist.test.images[:1000], mnist.test.labels[:1000]))
import tensorflow as tf
from tensorflow.keras import layers, models

# 1. 准备数据 (TF 2.0 内置了更方便的加载方式)
mnist = tf.keras.datasets.mnist
(x_train, y_train), (x_test, y_test) = mnist.load_data()
# 归一化并调整形状 [None, 28, 28, 1]
x_train, x_test = x_train / 255.0, x_test / 255.0
x_train = x_train[..., tf.newaxis].astype("float32")
x_test = x_test[..., tf.newaxis].astype("float32")

# 2. 构建模型 (Sequential 模式)
model = models.Sequential([
    # 卷积层 1: 7x7 核, 32个, padding same
    layers.Conv2D(32, (7, 7), activation='relu', padding='same', input_shape=(28, 28, 1)),
    layers.MaxPooling2D((2, 2), strides=2, padding='same'),
    
    # 卷积层 2: 5x5 核, 64个, padding same
    layers.Conv2D(64, (5, 5), activation='relu', padding='same'),
    layers.MaxPooling2D((2, 2), strides=2, padding='same'),
    
    # 全连接层
    layers.Flatten(),
    layers.Dense(1024, activation='relu'),
    layers.Dropout(0.3), # keep_prob 0.7 对应 dropout 0.3
    
    # 输出层
    layers.Dense(10, activation='softmax')
])

# 3. 编译模型
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
              loss='sparse_categorical_crossentropy', # 直接用类别索引作为标签
              metrics=['accuracy'])

# 4. 训练模型
model.fit(x_train, y_train, epochs=5, batch_size=100, validation_data=(x_test, y_test))

Epoch 1/5


c:\Users\64347\miniconda3\envs\tf_env\lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


600/600 ━━━━━━━━━━━━━━━━━━━━ 16s 25ms/step - accuracy: 0.8023 - loss: 0.7749 - val_accuracy: 0.9659 - val_loss: 0.1170
Epoch 2/5
600/600 ━━━━━━━━━━━━━━━━━━━━ 14s 24ms/step - accuracy: 0.9678 - loss: 0.1092 - val_accuracy: 0.9820 - val_loss: 0.0590
Epoch 3/5
600/600 ━━━━━━━━━━━━━━━━━━━━ 14s 23ms/step - accuracy: 0.9789 - loss: 0.0692 - val_accuracy: 0.9865 - val_loss: 0.0419
Epoch 4/5
600/600 ━━━━━━━━━━━━━━━━━━━━ 14s 23ms/step - accuracy: 0.9838 - loss: 0.0547 - val_accuracy: 0.9883 - val_loss: 0.0347
Epoch 5/5
600/600 ━━━━━━━━━━━━━━━━━━━━ 15s 24ms/step - accuracy: 0.9876 - loss: 0.0404 - val_accuracy: 0.9882 - val_loss: 0.0348
